In [0]:
import yaml
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from datetime import datetime, timedelta
from typing import Dict, List, Any, Tuple
import json

# Load YAML config from the same directory
config_path = "/Workspace/Users/ryean3@gmail.com/sephora-recommender/data_pipeline/(Clone) pipeline_health_config.yml"

print(f"📂 Loading config from: {config_path}")

try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    print("✅ Configuration loaded successfully!\n")
    print(f"   Bronze checks: {len(config.get('bronze_checks', {}))} tables")
    print(f"   Silver checks: {len(config.get('silver_checks', {}))} tables")
    print(f"   Gold checks: {len(config.get('gold_checks', {}))} tables")
    
    # Extract global config
    global_config = config.get('global', {})
    catalog = global_config.get('catalog', 'workspace')
    schema = global_config.get('schema', 'default')
    
    print(f"\n📊 Target catalog.schema: {catalog}.{schema}")
    
except FileNotFoundError:
    print(f"❌ Config file not found at {config_path}")
    print("   Please ensure pipeline_health_config.yml exists in the same directory.")
    raise
except yaml.YAMLError as e:
    print(f"❌ Error parsing YAML config: {e}")
    raise

In [0]:
class DataQualityChecker:
    def __init__(self, spark, catalog="workspace", schema="default"):
        self.spark = spark
        self.catalog = catalog
        self.schema = schema
        self.results = []
    
    def check_row_count(self, table_name: str, check_config: Dict) -> Dict:
        """Check if table has minimum number of rows"""
        df = self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")
        row_count = df.count()
        
        min_rows = check_config.get('min_rows', 0)
        alert_on_zero = check_config.get('alert_on_zero', False)
        
        passed = row_count >= min_rows
        if alert_on_zero and row_count == 0:
            passed = False
        
        return {
            'check_type': 'row_count',
            'passed': passed,
            'actual': row_count,
            'expected': f">= {min_rows}",
            'message': f"Row count: {row_count}" if passed else f"Row count {row_count} below minimum {min_rows}"
        }
    
    def check_schema_validation(self, table_name: str, check_config: Dict) -> Dict:
        """Validate required columns exist"""
        df = self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")
        actual_columns = set(df.columns)
        required_columns = set(check_config.get('required_columns', []))
        
        missing_columns = required_columns - actual_columns
        passed = len(missing_columns) == 0
        
        return {
            'check_type': 'schema_validation',
            'passed': passed,
            'actual': list(actual_columns),
            'expected': list(required_columns),
            'message': 'All required columns present' if passed else f"Missing columns: {missing_columns}"
        }
    
    def check_null_check(self, table_name: str, check_config: Dict) -> Dict:
        """Check null percentage in specified columns"""
        df = self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")
        columns = check_config.get('columns', [])
        max_null_percent = check_config.get('max_null_percent', 0)
        
        total_rows = df.count()
        results = []
        all_passed = True
        
        for col in columns:
            if col not in df.columns:
                continue
            
            null_count = df.filter(F.col(col).isNull()).count()
            null_percent = (null_count / total_rows * 100) if total_rows > 0 else 0
            
            passed = null_percent <= max_null_percent
            if not passed:
                all_passed = False
            
            results.append({
                'column': col,
                'null_percent': round(null_percent, 2),
                'passed': passed
            })
        
        return {
            'check_type': 'null_check',
            'passed': all_passed,
            'details': results,
            'message': 'Null checks passed' if all_passed else f"Null percentage exceeds {max_null_percent}% in some columns"
        }
    
    def check_duplicate_check(self, table_name: str, check_config: Dict) -> Dict:
        """Check for duplicate records based on key columns"""
        df = self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")
        key_columns = check_config.get('key_columns', [])
        max_duplicates = check_config.get('max_duplicates', 0)
        
        total_rows = df.count()
        distinct_rows = df.select(key_columns).distinct().count()
        duplicate_count = total_rows - distinct_rows
        
        passed = duplicate_count <= max_duplicates
        
        return {
            'check_type': 'duplicate_check',
            'passed': passed,
            'actual': duplicate_count,
            'expected': f"<= {max_duplicates}",
            'message': f"Found {duplicate_count} duplicates" if not passed else "No duplicates found"
        }
    
    def check_freshness(self, table_name: str, check_config: Dict) -> Dict:
        """Check if data is fresh based on timestamp column"""
        df = self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")
        timestamp_column = check_config.get('timestamp_column')
        max_age_hours = check_config.get('max_age_hours', 24)
        
        if timestamp_column not in df.columns:
            return {
                'check_type': 'freshness',
                'passed': False,
                'message': f"Timestamp column {timestamp_column} not found"
            }
        
        max_timestamp = df.agg(F.max(timestamp_column)).collect()[0][0]
        
        if max_timestamp is None:
            return {
                'check_type': 'freshness',
                'passed': False,
                'message': 'No timestamp data found'
            }
        
        age_hours = (datetime.now() - max_timestamp).total_seconds() / 3600
        passed = age_hours <= max_age_hours
        
        return {
            'check_type': 'freshness',
            'passed': passed,
            'actual': f"{age_hours:.1f} hours",
            'expected': f"<= {max_age_hours} hours",
            'message': f"Data is {age_hours:.1f} hours old" if not passed else "Data is fresh"
        }
    
    def check_value_range(self, table_name: str, check_config: Dict) -> Dict:
        """Check if column values are within expected range"""
        df = self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")
        column = check_config.get('column')
        min_value = check_config.get('min_value')
        max_value = check_config.get('max_value')
        
        # Cast to double for comparison
        df_typed = df.withColumn(column, F.col(column).cast('double'))
        
        out_of_range = df_typed.filter(
            (F.col(column) < float(min_value)) | (F.col(column) > float(max_value))
        ).count()
        
        passed = out_of_range == 0
        
        return {
            'check_type': 'value_range',
            'passed': passed,
            'actual': f"{out_of_range} records out of range",
            'expected': f"[{min_value}, {max_value}]",
            'message': f"Found {out_of_range} values outside range [{min_value}, {max_value}]"
        }
    
    def check_referential_integrity(self, table_name: str, check_config: Dict) -> Dict:
        """Check foreign key relationships"""
        df = self.spark.table(f"{self.catalog}.{self.schema}.{table_name}")
        column = check_config.get('column')
        reference_table = check_config.get('reference_table')
        reference_column = check_config.get('reference_column')
        
        ref_df = self.spark.table(f"{self.catalog}.{self.schema}.{reference_table}")
        
        # Find orphaned records
        orphaned = df.join(
            ref_df.select(F.col(reference_column).alias('ref_col')),
            df[column] == F.col('ref_col'),
            'left_anti'
        ).count()
        
        passed = orphaned == 0
        
        return {
            'check_type': 'referential_integrity',
            'passed': passed,
            'actual': f"{orphaned} orphaned records",
            'message': f"Found {orphaned} records without matching reference" if not passed else "Referential integrity maintained"
        }

print("✅ DataQualityChecker class defined")

In [0]:
def run_health_checks(layer: str = 'bronze'):
    """
    Run all health checks for specified layer
    """
    checker = DataQualityChecker(spark)
    check_config_key = f"{layer}_checks"
    
    if check_config_key not in config:
        print(f"❌ No checks configured for layer: {layer}")
        return
    
    layer_config = config[check_config_key]
    all_results = []
    
    print(f"\n{'='*60}")
    print(f"🔍 Running {layer.upper()} Layer Health Checks")
    print(f"{'='*60}\n")
    
    for table_name, table_config in layer_config.items():
        if not table_config.get('enabled', True):
            print(f"⏭️  Skipping {table_name} (disabled)")
            continue
        
        print(f"\n📊 Checking: {table_name}")
        print(f"   Description: {table_config.get('description', 'N/A')}")
        
        table_results = {
            'timestamp': datetime.now(),
            'layer': layer,
            'table_name': table_name,
            'checks': []
        }
        
        for check in table_config.get('checks', []):
            check_type = check.get('type')
            severity = check.get('severity', 'warning')
            
            try:
                # Route to appropriate check function
                if check_type == 'row_count':
                    result = checker.check_row_count(table_name, check)
                elif check_type == 'schema_validation':
                    result = checker.check_schema_validation(table_name, check)
                elif check_type == 'null_check':
                    result = checker.check_null_check(table_name, check)
                elif check_type == 'duplicate_check':
                    result = checker.check_duplicate_check(table_name, check)
                elif check_type == 'freshness':
                    result = checker.check_freshness(table_name, check)
                elif check_type == 'value_range':
                    result = checker.check_value_range(table_name, check)
                elif check_type == 'referential_integrity':
                    result = checker.check_referential_integrity(table_name, check)
                else:
                    result = {
                        'check_type': check_type,
                        'passed': False,
                        'message': f"Check type '{check_type}' not implemented"
                    }
                
                result['severity'] = severity
                table_results['checks'].append(result)
                
                # Print result
                status = "✅" if result['passed'] else "❌"
                severity_icon = "🔴" if severity == 'critical' else "⚠️"
                print(f"   {status} {check_type}: {result['message']} {severity_icon if not result['passed'] else ''}")
                
            except Exception as e:
                error_result = {
                    'check_type': check_type,
                    'passed': False,
                    'severity': severity,
                    'message': f"Error: {str(e)}"
                }
                table_results['checks'].append(error_result)
                print(f"   ❌ {check_type}: Error - {str(e)}")
        
        all_results.append(table_results)
    
    # Summary
    print(f"\n{'='*60}")
    print("📋 Summary")
    print(f"{'='*60}")
    
    total_checks = sum(len(r['checks']) for r in all_results)
    passed_checks = sum(sum(1 for c in r['checks'] if c['passed']) for r in all_results)
    failed_checks = total_checks - passed_checks
    
    critical_failures = sum(
        sum(1 for c in r['checks'] if not c['passed'] and c.get('severity') == 'critical')
        for r in all_results
    )
    
    print(f"Total Checks: {total_checks}")
    print(f"✅ Passed: {passed_checks}")
    print(f"❌ Failed: {failed_checks}")
    print(f"🔴 Critical Failures: {critical_failures}")
    
    return all_results

print("✅ Health check runner function defined")

In [0]:
# Run bronze layer health checks
bronze_results = run_health_checks('bronze')

In [0]:
# Save results to Delta table for tracking
def save_results_to_table(results: List[Dict]):
    """
    Save health check results to Delta table
    """
    results_table = config.get('reporting', {}).get('results_table', 'workspace.default.data_quality_results')
    
    # Flatten results for storage
    flattened = []
    for table_result in results:
        for check in table_result['checks']:
            flattened.append({
                'timestamp': table_result['timestamp'],
                'layer': table_result['layer'],
                'table_name': table_result['table_name'],
                'check_type': check['check_type'],
                'passed': check['passed'],
                'severity': check.get('severity', 'warning'),
                'message': check.get('message', ''),
                'details': json.dumps(check)
            })
    
    # Create DataFrame
    results_df = spark.createDataFrame(flattened)
    
    # Save to Delta
    results_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(results_table)
    
    print(f"\n✅ Results saved to {results_table}")
    return results_df

# Save bronze results
if bronze_results:
    results_df = save_results_to_table(bronze_results)
    display(results_df)